# Train "my-ai-coder" (Qwen3.5 4B) on the free Colab GPU

**Before you start (one time, ~2 minutes):**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Get a free Hugging Face token: huggingface.co → Settings → Access Tokens → **Create new token** → type **Write** → copy it
3. In Colab, click the **🔑 key icon** (left sidebar) → **Add new secret** → Name: `HF_TOKEN`, Value: paste token → turn on **Notebook access**

**Then:** Runtime → **Run all**. Takes roughly 1–2 hours. Keep the tab open.

When it finishes, your trained model is on Hugging Face and your PC can download it with one command (shown at the end).

> Why 4B and not 9B: training 9B needs ~22 GB of GPU memory; the free T4 has ~15 GB.

## 1. Settings (change these if you want)

In [ ]:
HF_USERNAME   = "Hudson2011"            # your Hugging Face username
REPO_NAME     = "my-ai-coder-GGUF"      # where the trained model gets uploaded
BASE_MODEL    = "unsloth/Qwen3.5-4B"
NUM_EXAMPLES  = 3000     # coding examples to train on (more = longer + stronger)
MAX_STEPS     = 300      # training steps (~1-2 h on a T4)
MAX_SEQ_LEN   = 4096     # longest example kept (in tokens)

## 2. Install Unsloth (2–4 minutes)

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")   # fails here if the secret wasn't added (see step 3 at top)
print("HF token found.")

## 3. Load the base model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,   # Unsloth advises against 4-bit training for Qwen3.5
    load_in_16bit  = True,
)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)   # text part of the tokenizer

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LEN,
)

## 4. Quick test BEFORE training (to compare later)

In [ ]:
TEST_PROMPTS = [
    "Write a Python function that returns the longest palindromic substring of a string. Explain the time complexity.",
    "In JavaScript, why does `[1,2,10].sort()` give [1,10,2]? Show the fix.",
    "What causes inflation, in 3 sentences?",   # general-knowledge check: should NOT get worse
]
def ask(prompt, max_new_tokens=700):
    FastLanguageModel.for_inference(model)
    msgs = [{"role": "user", "content": prompt}]
    ids = text_tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, top_k=20)
    FastLanguageModel.for_training(model)
    return text_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

before = {p: ask(p) for p in TEST_PROMPTS}
for p, a in before.items(): print("Q:", p, "\n--- BEFORE ---\n", a[-1500:], "\n" + "="*80)

## 5. Get coding training data (streamed, so the 28 GB dataset is never fully downloaded)

In [ ]:
from datasets import load_dataset, Dataset

stream = load_dataset("nvidia/OpenCodeReasoning", "split_0", split="split_0", streaming=True)
stream = stream.shuffle(seed=3407, buffer_size=10_000)

rows = []
for ex in stream:
    msgs = [{"role": "user", "content": ex["input"]},
            {"role": "assistant", "content": ex["output"]}]   # output keeps its <think> reasoning
    text = text_tok.apply_chat_template(msgs, tokenize=False)
    if len(text_tok(text).input_ids) <= MAX_SEQ_LEN:          # skip examples that are too long
        rows.append({"text": text})
    if len(rows) >= NUM_EXAMPLES: break

dataset = Dataset.from_list(rows)
print(len(dataset), "examples ready. Sample:\n", dataset[0]["text"][:800])

## 6. Train (the long part, about 1–2 hours). Loss should trend down.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = 1e-4,          # gentle, to limit forgetting general knowledge
        logging_steps = 10,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        output_dir = "outputs",
        save_steps = 100,              # checkpoints, in case Colab disconnects
        seed = 3407,
        report_to = "none",
        dataset_num_proc = 1,
    ),
)
stats = trainer.train()
print(stats)

## 7. Test AFTER training. Compare with section 4. If it got worse, don't use it.

In [ ]:
for p in TEST_PROMPTS:
    print("Q:", p, "\n--- AFTER ---\n", ask(p)[-1500:], "\n" + "="*80)

## 8. Save to Hugging Face as a GGUF file Ollama can run (10–20 minutes)

In [ ]:
repo = f"{HF_USERNAME}/{REPO_NAME}"
model.push_to_hub_gguf(repo, tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("Uploaded to https://huggingface.co/" + repo)

## 9. Done. On your PC, run:

```
ollama pull hf.co/Hudson2011/my-ai-coder-GGUF:Q4_K_M
ollama run hf.co/Hudson2011/my-ai-coder-GGUF:Q4_K_M
```

(If the tag errors, open the Hugging Face repo page and use the exact file name shown there.)

**Tips**
- Colab disconnected mid-training? Run all again; it starts over (free tier limit).
- Want it stronger? Raise `NUM_EXAMPLES` and `MAX_STEPS`, but watch the general-knowledge test.